# MCMC Portfolio Demo: Parameter Dynamics and Benchmarks

This notebook is designed for portfolio presentation. It demonstrates:

1. how the MCMC parameters evolve across iterations,
2. whether acceptance rates look reasonable,
3. how the MCMC preference model compares with simple benchmarks on held-out test sessions.

The data is synthetic and reproducible. The modeling pipeline is compatible with real browsing logs using the same schema.

## 1. Setup

If running on Colab, choose `Runtime > Change runtime type > GPU` before running the GPU-parallel script. The script uses CuPy when available and falls back to NumPy otherwise.

In [ ]:
# Colab setup, if needed:
# !pip install pyarrow scipy seaborn matplotlib tqdm
# !pip install cupy-cuda12x  # optional, only for GPU runtime

from pathlib import Path
import json
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import difflib

sns.set_theme(style="whitegrid")

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "mcmc_gpu_parallel.py").exists() and (PROJECT_DIR / "mcmc_portfolio").exists():
    PROJECT_DIR = PROJECT_DIR / "mcmc_portfolio"

DATA_PATH = PROJECT_DIR / "data" / "itemPV_202201to202204.parquet"
OUTPUT_DIR = PROJECT_DIR / "outputs_demo"
RESULT_PATH = OUTPUT_DIR / "gpu_mcmc_results.npz"
SUMMARY_PATH = OUTPUT_DIR / "gpu_mcmc_summary.json"

TOP10 = [5, 9, 19, 11, 20, 23, 6, 21, 25, 22]
CATEGORY_LABELS = {
    0: "Toys",
    1: "Home",
    2: "Vehicles",
    3: "Electronics",
    4: "Travel",
    5: "Appliances",
    6: "Books",
    7: "Mobile",
    8: "Lifestyle",
    9: "Gaming",
}

PROJECT_DIR, DATA_PATH


## 2. Run or Load MCMC Samples

For a quick demo, start with a small setting. Increase `N_USERS` and `N_ITER` for the final portfolio version.

In [ ]:
N_USERS = 500
N_ITER = 50
SEED = 123

if not RESULT_PATH.exists():
    cmd = [
        sys.executable,
        str(PROJECT_DIR / "mcmc_gpu_parallel.py"),
        "--data", str(DATA_PATH),
        "--n-users", str(N_USERS),
        "--n-iter", str(N_ITER),
        "--seed", str(SEED),
        "--output-dir", str(OUTPUT_DIR),
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    print("Loading existing results:", RESULT_PATH)

results = np.load(RESULT_PATH)
summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))
summary

## 3. Acceptance Rates and Predictive Accuracy

Acceptance rates help diagnose whether the Metropolis-Hastings proposal scale is too aggressive or too conservative. The accuracy trace gives a quick view of predictive behavior during sampling.

In [ ]:
accept_trace = results["accept_trace"]
accuracy_trace = results["accuracy_trace"]
iterations = np.arange(1, len(accuracy_trace) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(iterations, accept_trace[:, 0], label=r"$a_u$")
axes[0].plot(iterations, accept_trace[:, 1], label=r"$s_u$")
axes[0].plot(iterations, accept_trace[:, 2], label=r"$C^{switch}_u$")
axes[0].set_title("Acceptance Rate by Parameter Block")
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("Acceptance Rate")
axes[0].legend()

axes[1].plot(iterations, accuracy_trace, color="tab:green")
axes[1].set_title("Training Next-Category Accuracy")
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("Accuracy")
plt.tight_layout()

## 4. Hyperparameter Trace

The hyperparameter trace summarizes population-level preference movement during sampling.

In [ ]:
theta_a_trace = results["theta_a_trace"]
theta_s_trace = results["theta_s_trace"]
theta_switch_trace = results["theta_switch_trace"]

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
for k in range(theta_a_trace.shape[1]):
    axes[0].plot(iterations, theta_a_trace[:, k], label=CATEGORY_LABELS[k], alpha=0.85)
axes[0].set_title(r"Population Preference Trace: $\theta_a$")
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel(r"$\theta_{a,k}$")
axes[0].legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)

axes[1].plot(iterations, theta_s_trace, label=r"$\theta_s$")
axes[1].plot(iterations, theta_switch_trace, label=r"$\theta_{switch}$")
axes[1].set_title("Scalar Hyperparameter Trace")
axes[1].set_xlabel("Iteration")
axes[1].legend()
plt.tight_layout()

## 5. User-Level Parameter Dynamics

Only a few example users are plotted to keep the demo readable. This shows how individual preference, diversity, and switching-cost parameters move during sampling.

In [ ]:
demo_user_ids = results["demo_user_ids"]
demo_a_trace = results["demo_a_trace"]
demo_s_trace = results["demo_s_trace"]
demo_switch_trace = results["demo_switch_trace"]

user_i = 0
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for k in range(demo_a_trace.shape[2]):
    axes[0].plot(iterations, demo_a_trace[:, user_i, k], label=CATEGORY_LABELS[k], alpha=0.85)
axes[0].set_title(rf"User {demo_user_ids[user_i]} Preference Trace: $a_u$")
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel(r"$a_{ku}$")

for j, uid in enumerate(demo_user_ids):
    axes[1].plot(iterations, demo_s_trace[:, j], label=f"user {uid}")
axes[1].set_title(r"Diversity / Elasticity Trace: $s_u=1+z_u$")
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel(r"$s_u$")

for j, uid in enumerate(demo_user_ids):
    axes[2].plot(iterations, demo_switch_trace[:, j], label=f"user {uid}")
axes[2].set_title(r"Switching Cost Trace: $C^{switch}_u$")
axes[2].set_xlabel("Iteration")
axes[2].set_ylabel(r"$C^{switch}_u$")
axes[2].legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()

## 6. Posterior Summary

Final posterior samples can be summarized as user heterogeneity. In a longer run, use burn-in and posterior averages rather than only the final draw.

In [ ]:
final_a = results["final_a"]
final_s = results["final_s"]
final_switch = results["final_switch"]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].hist(final_a.mean(axis=1), bins=30, color="tab:blue", alpha=0.8)
axes[0].set_title(r"Mean Preference Strength: $\bar{a}_u$")
axes[0].set_xlabel(r"$\bar{a}_u$")

axes[1].hist(final_s, bins=30, color="tab:orange", alpha=0.8)
axes[1].set_title(r"Diversity Parameter: $s_u$")
axes[1].set_xlabel(r"$s_u$")

axes[2].hist(final_switch, bins=30, color="tab:red", alpha=0.8)
axes[2].set_title(r"Switching Cost: $C^{switch}_u$")
axes[2].set_xlabel(r"$C^{switch}_u$")
plt.tight_layout()

## 7. Benchmark Models

The comparison uses held-out test sessions. The models are:

- **Global Popularity**: always predicts the most frequent training categories.
- **User Most Frequent**: predicts each user's most frequent training categories.
- **Markov Transition**: predicts from the previous category using the empirical transition matrix.
- **MCMC Preference Model**: predicts using final estimated user parameters.

In [ ]:
def prepare_train_test(df):
    df = df.copy()
    df["gclass"] = df["g_class4"].astype(int)
    df = df[df["gclass"].isin(TOP10)].copy()
    map_dict = {key: value for value, key in enumerate(TOP10)}
    df["c"] = df["gclass"].map(map_dict).astype(int)
    train = df[df["time"] < "2022-02-01"].copy()
    test = df[df["time"] >= "2022-02-01"].copy()
    selected_users = set(results["selected_users"].astype(int))
    train = train[train["CID"].isin(selected_users)].copy()
    test = test[test["CID"].isin(selected_users)].copy()
    return train, test


def transition_matrix_from_train(train):
    trans = np.zeros((10, 10), dtype=float)
    seqs = train.sort_values(["CID", "session_id", "order"]).groupby(["CID", "session_id"])["c"].apply(list)
    for seq in seqs:
        for prev, nxt in zip(seq[:-1], seq[1:]):
            trans[prev, nxt] += 1
    denom = trans.sum(axis=1, keepdims=True)
    return np.divide(trans, denom, out=np.zeros_like(trans), where=denom != 0)


def mcmc_utility(a, s, c_switch, state, switch_row):
    power1 = s / (s - 1)
    power2 = 1 / (s - 1)
    power3 = -1 / s
    if state.sum() == 0:
        return a ** power1
    term = np.sum(a * (state ** power1))
    return (max(term, 1e-12) ** power2) * a * ((state + 1) ** power3) - c_switch * switch_row


def topk_from_scores(scores, k=3):
    return np.argsort(scores)[::-1][:k].tolist()


def evaluate_benchmarks(train, test):
    selected_users = results["selected_users"].astype(int)
    user_index = {cid: i for i, cid in enumerate(selected_users)}
    final_a = results["final_a"]
    final_s = results["final_s"]
    final_switch = results["final_switch"]
    switching_cost = results["switching_cost"]

    global_counts = train["c"].value_counts().reindex(range(10), fill_value=0).to_numpy()
    global_top = topk_from_scores(global_counts, 3)

    user_counts = (
        train.groupby(["CID", "c"]).size().unstack(fill_value=0).reindex(columns=range(10), fill_value=0)
    )
    trans = transition_matrix_from_train(train)

    metrics = {
        name: {"top1": [], "top3": [], "seq_true": [], "seq_pred": []}
        for name in ["Global Popularity", "User Most Frequent", "Markov Transition", "MCMC Preference Model"]
    }

    grouped = test.sort_values(["CID", "session_id", "order"]).groupby(["CID", "session_id"])
    for (cid, session_id), session in grouped:
        state = np.zeros(10, dtype=float)
        prev = None
        session_true = []
        session_pred = {name: [] for name in metrics}

        for true_c in session["c"].to_numpy():
            preds = {}
            preds["Global Popularity"] = global_top

            if cid in user_counts.index:
                preds["User Most Frequent"] = topk_from_scores(user_counts.loc[cid].to_numpy(), 3)
            else:
                preds["User Most Frequent"] = global_top

            if prev is None or trans[prev].sum() == 0:
                preds["Markov Transition"] = global_top
            else:
                preds["Markov Transition"] = topk_from_scores(trans[prev], 3)

            u = user_index[cid]
            switch_row = np.zeros(10) if prev is None else switching_cost[prev]
            scores = mcmc_utility(final_a[u], final_s[u], final_switch[u], state, switch_row)
            preds["MCMC Preference Model"] = topk_from_scores(scores, 3)

            for name, pred_top3 in preds.items():
                metrics[name]["top1"].append(pred_top3[0] == true_c)
                metrics[name]["top3"].append(true_c in pred_top3)
                session_pred[name].append(pred_top3[0])

            session_true.append(true_c)
            state[true_c] += 1
            prev = true_c

        for name in metrics:
            metrics[name]["seq_true"].append(session_true)
            metrics[name]["seq_pred"].append(session_pred[name])

    rows = []
    for name, item in metrics.items():
        seq_sim = [
            difflib.SequenceMatcher(None, pred, true).ratio()
            for pred, true in zip(item["seq_pred"], item["seq_true"])
        ]
        rows.append({
            "Model": name,
            "Top-1 Accuracy": np.mean(item["top1"]),
            "Top-3 Accuracy": np.mean(item["top3"]),
            "Sequence Similarity": np.mean(seq_sim),
        })
    return pd.DataFrame(rows).sort_values("Top-1 Accuracy", ascending=False)


df = pd.read_parquet(DATA_PATH)
train, test = prepare_train_test(df)
benchmark_df = evaluate_benchmarks(train, test)
benchmark_df

## 8. Benchmark Visualization

This chart is usually the clearest portfolio slide: it shows whether the preference model adds value beyond simple rules.

In [ ]:
plot_df = benchmark_df.melt(id_vars="Model", var_name="Metric", value_name="Score")
plt.figure(figsize=(12, 5))
sns.barplot(data=plot_df, x="Metric", y="Score", hue="Model")
plt.title("Held-Out Test Benchmark Comparison")
plt.ylim(0, 1)
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()

## 9. Suggested Portfolio Interpretation

A concise interpretation you can use:

> The MCMC model estimates user-level preference strength, diversity, and switching-cost parameters from training browsing sessions. Parameter traces show the iterative posterior sampling process, while held-out benchmark comparisons evaluate whether the learned preference model improves next-category prediction over popularity, user-frequency, and Markov baselines.